# 대화당 비용 백분위수

Python 3.12 새 커널에서 대화당 합성 비용 100개 준비부터 확인합니다.

## 개념을 코드에 연결하기

**호출 표기와 실행 시점.** 대화당 값은 한 대화에 속한 모든 모델 호출의 사용량을 합쳐 만듭니다. 값을 오름차순으로 정렬한 뒤 p50은 중앙 수준, p95는 95%의 대화가 그 이하인 경계, p99는 극단부에 가까운 경계를 나타냅니다. 이 절은 가장 가까운 순위 방법을 사용하므로 계산법도 결과와 함께 기록합니다.

**다른 상황에 적용.** 대부분 한 번만 답하지만 일부 요청은 재작성과 재시도를 거치는 서비스에서는 상위 백분위수에서 긴 꼬리가 나타날 수 있습니다. 이 표본에서 p95는 일반 구간의 상단이고 긴 꼬리는 p99에서 나타납니다. 기능별 태그로 나누면 어떤 경로가 상위 백분위수를 만드는지 찾을 수 있습니다.

**비교와 경계.** 백분위수는 원인을 설명하지 않습니다. p99가 높다면 trace에서 호출 수, 입력 길이, 재시도 여부를 다시 봐야 합니다. 표본 100개의 합성 분포는 실제 사용자 분포나 미래 가격을 나타내지 않습니다.


## Python 3.12 패키지 버전 고정

**이 셀의 코드 읽기.** `%pip install`은 이 노트북 커널이 사용할 패키지를 설치합니다. 앞 셀의 값을 입력으로 사용하지 않으며, 설치가 끝난 뒤의 코드 셀에서 패키지를 가져와 사용합니다.


In [1]:
# 실습 환경에서 사용하는 보조 패키지 버전을 고정합니다.
%pip install -q python-dotenv==1.2.3


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


## 대화당 합성 비용 100개 준비

**이 셀의 코드 읽기.** `for` 블록은 입력 항목을 하나씩 처리하며, 반복이 끝난 뒤 모은 값이 다음 셀의 입력이 됩니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [2]:
target_costs = [0.002 + (i % 7) * 0.001 for i in range(95)]
target_costs += [0.020, 0.035, 0.050, 0.080, 0.120]
call_rows = []
for index, total in enumerate(target_costs):
    conversation_id = f'conv-{index:03d}'
    call_rows.append({'conversation_id': conversation_id, 'cost_usd': total * 0.4})
    call_rows.append({'conversation_id': conversation_id, 'cost_usd': total * 0.6})
cost_by_conversation = {}
for row in call_rows:
    key = row['conversation_id']
    cost_by_conversation[key] = cost_by_conversation.get(key, 0.0) + row['cost_usd']
costs = list(cost_by_conversation.values())
print('호출 행:', len(call_rows), '대화 수:', len(costs))
print('첫 대화 합계:', round(costs[0], 3))
print('총비용(교육용 달러):', round(sum(costs), 3))
assert len(call_rows) == 200 and len(costs) == 100

호출 행: 200 대화 수: 100
첫 대화 합계: 0.002
총비용(교육용 달러): 0.774


## 가장 가까운 순위로 p50·p95·p99 계산

**이 셀의 코드 읽기.** `nearest_rank(values: list[float], percentile: int)`는 함수를 정의하는 표기입니다. 이 셀에서는 함수 객체만 준비되고 들여쓴 본문은 뒤에서 함수를 호출할 때 실행됩니다. `return` 뒤의 값은 화면 출력이 아니라 호출한 코드로 전달되는 반환값입니다. `nearest_rank(values, percentile)`은 값을 정렬한 뒤 지정 백분위의 가장 가까운 순위 값을 반환합니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다.


In [3]:
# 정렬한 비용에서 가장 가까운 순위 방식으로 p50·p95·p99를 계산합니다.
import math, statistics
def nearest_rank(values: list[float], percentile: int) -> float:
    ordered = sorted(values)
    rank = max(1, math.ceil(percentile / 100 * len(ordered)))
    return ordered[rank - 1]
for p in (50, 95, 99):
    print(f'p{p}:', f'${nearest_rank(costs, p):.3f}')
print('평균:', f'${statistics.mean(costs):.3f}')

p50: $0.005
p95: $0.008
p99: $0.080
평균: $0.008


## p99와 p50의 비율 해석

**이 셀의 코드 읽기.** `nearest_rank(values, percentile)`은 값을 정렬한 뒤 지정 백분위의 가장 가까운 순위 값을 반환합니다. `print()`는 확인용 문자열을 화면에 표시할 뿐 함수의 반환값을 바꾸지 않습니다. `assert`는 조건이 참이면 출력 없이 통과하고 거짓이면 실행을 중단하므로, 바로 앞 결과를 검증하는 역할을 합니다.


In [4]:
# p99와 p50의 비율로 긴 꼬리 비용이 존재하는지 확인합니다.
p50 = nearest_rank(costs, 50)
p99 = nearest_rank(costs, 99)
print('p99/p50:', round(p99 / p50, 1))
assert p99 > p50 * 10
print('해석: 일부 대화의 긴 꼬리 비용을 별도로 조사합니다.')

p99/p50: 16.0
해석: 일부 대화의 긴 꼬리 비용을 별도로 조사합니다.
